# Induction Evals
We evaluate that the induction case is strong enough to drive a wedge between LLM performance in this notebook.

In [1]:
"""The following generates the Quiz all our models will be evaluated on."""

import itertools
import string
import numpy as np

import logging

from ordered_set import OrderedSet
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
load_dotenv("./keys.env", verbose=True)

from smolbench.induction.chromatic import (
    ChromaticIntervalsConfig,
    Prompter,
    get_random_exclusive_quiz,
    anneal_intervals,
)
from smolbench.evals import Marks
from smolbench.evals.openrouter import evaluate

template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. The following lists the people who were $role and"
    " the years they were $role:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "Between the years $start and $end, exclusive of the end, could $color"
    " have headed the $parade parade every year?"
)


def query_gen(
    labels_to_intervals: Dict[Color, Intervals],
    interval_to_label: Dict[Intervals, Color],
    seed: int,
) -> Dict[str, str]:
    """Generates a series of queries"""
    rng: np.random.Generator = np.random.default_rng(seed)
    # Finds max interval.
    n: int = max(interval[1] for interval in interval_to_label)
    for color, intervals in labels_to_intervals.items():
        # Generates a series of true items.
        for start, end in intervals:
            start, end = np.sort(
                rng.choice(range(start, end + 1), size=2, replace=False)
            )
            yield {"color": color, "start": start, "end": end}, True
        # Generates a false proposition.
        invalid_range: Intervals = anneal_intervals(
            itertools.chain(
                (
                    OrderedSet(interval_to_label.keys())
                    - OrderedSet(itertools.chain(*intervals))
                )
            )
        )
        for start, end in invalid_range:
            start = rng.choice(range(start, end))
            # Binom with p = intervals / n capped at end for a similar-ish
            # distr. to positive accounts.
            end = min(
                end,
                start
                + rng.binomial(
                    end - start + 1,
                    np.mean([len(interval) for interval in interval_to_label]) / n,
                )
                + 1,
            )
            yield {"color": color, "start": start, "end": end}, False


SEED: int = 1776
intens_quiz, extens_quiz = get_random_exclusive_quiz(
    ChromaticIntervalsConfig(
        n=12 * 250,
        intervals= 250 // 4,
        colors=45,
        seed=SEED,
    ),
    Prompter(
        template,
        {
            "role": "Twislax",
            "parade": "Gildane",
        },
        query_gen,
    ),
)

## Prompt Validation

In [ ]:
print(intens_quiz[0].prompt)

In [ ]:
print(extens_quiz[0].prompt)

# Decoder-Only Model
This section tests classical decoder-only models.

In [2]:
decode_intens_eval: Marks = evaluate(
    intens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED
)

INFO:root:Response:
{'id': 'gen-1775084104-FbxGHLriBIP07pE0HSdi', 'object': 'chat.completion', 'created': 1775084104, 'model': 'mistralai/mistral-7b-instruct-v0.1', 'provider': 'Cloudflare', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': ' False', 'refusal': None, 'reasoning': None}}], 'usage': {'prompt_tokens': 1170, 'completion_tokens': 2, 'total_tokens': 1172, 'cost': 0.00012908, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00012908, 'upstream_inference_prompt_cost': 0.0001287, 'upstream_inference_completions_cost': 3.8e-07}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}}
 was 1172 <= 2824
INFO:root:Response:
{'id': 'gen-1775084104-kb9f6oSVKKNLmssqzBmn', 'object': 'chat.completion', 'crea

In [3]:
decode_extens_eval: Marks = evaluate(
    extens_quiz, "mistralai/mistral-7b-instruct-v0.1", SEED
)

INFO:root:Response:
{'id': 'gen-1775084198-vke17IiL8rGGnrr9rZYg', 'object': 'chat.completion', 'created': 1775084198, 'model': 'mistralai/mistral-7b-instruct-v0.1', 'provider': 'Cloudflare', 'system_fingerprint': None, 'choices': [{'index': 0, 'logprobs': None, 'finish_reason': 'stop', 'native_finish_reason': 'stop', 'message': {'role': 'assistant', 'content': ' False', 'refusal': None, 'reasoning': None}}], 'usage': {'prompt_tokens': 1459, 'completion_tokens': 2, 'total_tokens': 1461, 'cost': 0.00016087, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00016087, 'upstream_inference_prompt_cost': 0.00016049, 'upstream_inference_completions_cost': 3.8e-07}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}}
 was 1461 <= 2824
INFO:root:Response:
{'id': 'gen-1775084198-miX6IFlSHvMtRgfl7nON', 'object': 'chat.completion', 'cre

In [4]:
print(
    decode_intens_eval.correct, decode_intens_eval.incorrect, decode_intens_eval.invalid
)
print(
    decode_extens_eval.correct, decode_extens_eval.incorrect, decode_extens_eval.invalid
)

38 56 13
25 41 41


## CoT Model
This section tests a CoT model.

In [ ]:
cot_intens_eval: Marks = evaluate(
    intens_quiz, "qwen/qwen3-14b", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [ ]:
cot_extens_eval: Marks = evaluate(
    extens_quiz, "qwen/qwen3-14b", SEED, extra_args={"reasoning": {"enabled": True}}
)

In [13]:
print(cot_intens_eval.correct, cot_intens_eval.incorrect, cot_intens_eval.invalid)
print(cot_extens_eval.correct, cot_extens_eval.incorrect, cot_extens_eval.invalid)

107 0 0
104 3 0


## MoE Model
This section tests an MoE model.

In [ ]:
moe_intens_eval: Marks = evaluate(intens_quiz, "openai/gpt-oss-20b", SEED)

In [ ]:
moe_extens_eval: Marks = evaluate(extens_quiz, "openai/gpt-oss-20b", SEED)

In [12]:
print(moe_intens_eval.correct, moe_intens_eval.incorrect, moe_intens_eval.invalid)
print(moe_extens_eval.correct, moe_extens_eval.incorrect, moe_extens_eval.invalid)

106 1 0
100 7 0


# HRM Model
This section tests an HRM model.

In [ ]:
hrm_intens_eval: Marks = evaluate(intens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
hrm_extens_eval: Marks = evaluate(extens_quiz, "perplexity/pplx-embed-v1-4b", SEED)

In [ ]:
print(hrm_intens_eval.correct, hrm_intens_eval.incorrect, hrm_intens_eval.invalid)
print(hrm_extens_eval.correct, hrm_extens_eval.incorrect, hrm_extens_eval.invalid)